# Parameter-Efficient Fine-Tuning (PEFT)

Hands-on examples for Module 06, covering:

1. LoRA configuration and training
2. QLoRA with 4-bit quantization
3. Rank ablation study
4. Adapter merging and multi-adapter scenarios
5. Memory profiling
6. Complete PEFT pipeline

## Setup

In [ ]:
# Install required packages
# !pip install transformers datasets accelerate peft bitsandbytes torch

import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. LoRA Configuration Basics

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# Use a small model for demonstration
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load base model
print(f"Loading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Count parameters before LoRA
total_params = sum(p.numel() for p in model.parameters())
print(f"\nBase model parameters: {total_params:,}")

In [ ]:
# Configure LoRA with different ranks
def create_lora_config(rank=16, alpha=None, dropout=0.05):
    """Create LoRA config with sensible defaults."""
    if alpha is None:
        alpha = rank * 2  # Standard scaling
    
    return LoraConfig(
        r=rank,
        lora_alpha=alpha,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

# Compare different ranks
ranks = [4, 8, 16, 32, 64]
param_counts = []

print("LoRA Parameter Comparison:")
print("-" * 50)

for r in ranks:
    lora_config = create_lora_config(rank=r)
    lora_model = get_peft_model(model, lora_config)
    
    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in lora_model.parameters())
    
    param_counts.append(trainable)
    print(f"r={r:2d}: Trainable: {trainable:>10,} ({100*trainable/total:.3f}%)")
    
    # Clean up for next iteration
    del lora_model
    torch.cuda.empty_cache()

# Plot parameter count vs rank
plt.figure(figsize=(10, 5))
plt.bar([str(r) for r in ranks], param_counts, color='#238636')
plt.xlabel("LoRA Rank (r)")
plt.ylabel("Trainable Parameters")
plt.title("LoRA Trainable Parameters by Rank")
plt.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(param_counts):
    plt.text(i, v + 5000, f"{v:,}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 2. QLoRA with 4-bit Quantization

In [ ]:
# Check if bitsandbytes is available
try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
    print("bitsandbytes available - QLoRA enabled")
except ImportError:
    BNB_AVAILABLE = False
    print("bitsandbytes not available - skipping QLoRA demo")
    print("Install with: pip install bitsandbytes")

In [ ]:
if BNB_AVAILABLE and torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    
    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",      # NormalFloat 4-bit
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,  # Double quantization
    )
    
    print("Loading model with 4-bit quantization...")
    model_4bit = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    
    # Prepare for k-bit training
    model_4bit = prepare_model_for_kbit_training(model_4bit)
    
    # Apply LoRA
    lora_config = create_lora_config(rank=16)
    qlora_model = get_peft_model(model_4bit, lora_config)
    qlora_model.print_trainable_parameters()
    
    # Memory comparison
    print("\nMemory Comparison (7B model reference):")
    print("-" * 50)
    print("Full Fine-Tuning (16-bit):  ~28 GB")
    print("LoRA (16-bit):              ~6 GB")
    print("QLoRA (4-bit):              ~3 GB")
    print(f"\nActual VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Rank Ablation Study

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer

# Load small dataset for ablation study
dataset = load_dataset("mlabonne/FineTome-100k", split="train[:500]")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text", "question", "response"])
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Dataset: {len(tokenized_dataset)} samples")

In [ ]:
# Run ablation study with different ranks
def train_with_rank(rank, train_steps=50):
    """Train LoRA model with specified rank and return final loss."""
    # Fresh base model
    base = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    
    # Apply LoRA
    lora_config = create_lora_config(rank=rank)
    lora_model = get_peft_model(base, lora_config)
    
    # Train
    training_args = TrainingArguments(
        output_dir=f"./lora-r{rank}",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        max_steps=train_steps,
        logging_steps=train_steps,
        report_to="none",
        fp16=False,
    )
    
    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=tokenized_dataset,
    )
    
    trainer.train()
    final_loss = trainer.state.log_history[-1].get('loss', float('inf'))
    
    # Clean up
    del lora_model, base
    torch.cuda.empty_cache()
    
    return final_loss

# Run ablation (this takes a few minutes)
print("Running rank ablation study...")
print("This will train with r=4, 8, 16, 32 and compare final losses.")

ablation_ranks = [4, 8, 16, 32]
ablation_losses = []

for r in ablation_ranks:
    print(f"\nTraining with r={r}...")
    loss = train_with_rank(r)
    ablation_losses.append(loss)
    print(f"Final loss: {loss:.4f}")

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(ablation_ranks, ablation_losses, 'bo-', linewidth=2, markersize=8)
plt.xlabel("LoRA Rank")
plt.ylabel("Final Training Loss")
plt.title("Effect of Rank on Training Loss")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar([str(r) for r in ablation_ranks], ablation_losses, color='#238636')
plt.xlabel("LoRA Rank")
plt.ylabel("Final Training Loss")
plt.title("Loss by Rank")
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Find optimal rank (where loss plateaus)
loss_diffs = np.diff(ablation_losses)
plateau_idx = np.where(np.abs(loss_diffs) < 0.05)[0]
if len(plateau_idx) > 0:
    optimal_rank = ablation_ranks[plateau_idx[0] + 1]
    print(f"\nOptimal rank (loss plateau): r={optimal_rank}")
else:
    print(f"\nLoss still decreasing at r={ablation_ranks[-1]}, consider testing higher ranks")

## 4. Adapter Merging and Multi-Adapter

In [ ]:
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

# Create first adapter (simulate "domain" adapter)
lora_config_1 = create_lora_config(rank=8)
model = get_peft_model(base_model, lora_config_1, adapter_name="domain")

# Add second adapter (simulate "task" adapter)
lora_config_2 = create_lora_config(rank=8)
model.add_adapter("task", lora_config_2)

print("Available adapters:", model.adapter_names)

# Switch between adapters
model.set_adapter("domain")
print(f"\nActive adapter: {model.active_adapter}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

model.set_adapter("task")
print(f"\nActive adapter: {model.active_adapter}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Merge adapter into base model (for deployment)
print("Merging adapter into base model...")
model.set_adapter("domain")
merged_model = model.merge_and_unload()

print(f"Merged model parameters: {sum(p.numel() for p in merged_model.parameters()):,}")
print(f"Trainable params after merge: {sum(p.numel() for p in merged_model.parameters() if p.requires_grad):,}")
print("(All params are now trainable since adapter is merged)")

# Save merged model
# merged_model.save_pretrained("./merged-model")
# tokenizer.save_pretrained("./merged-model")

## 5. Memory Profiling

In [ ]:
def profile_memory(model, batch_size=4, seq_length=256):
    """Profile peak memory usage during forward pass."""
    if not torch.cuda.is_available():
        print("CUDA not available - skipping memory profiling")
        return
    
    # Create dummy input
    input_ids = torch.randint(100, 10000, (batch_size, seq_length)).cuda()
    attention_mask = torch.ones_like(input_ids)
    
    # Reset and measure
    torch.cuda.reset_peak_memory_stats()
    
    with torch.no_grad():
        model(input_ids=input_ids, attention_mask=attention_mask)
    
    peak_memory = torch.cuda.max_memory_allocated() / 1e9
    
    print(f"Peak memory (batch={batch_size}, seq={seq_length}): {peak_memory:.2f} GB")
    
    # Estimate max batch size for 8GB VRAM
    estimated_max_batch = int(8 / peak_memory * batch_size)
    print(f"Estimated max batch for 8GB VRAM: {estimated_max_batch}")
    
    return peak_memory

# Profile different configurations
print("Memory Profiling:")
print("=" * 50)

# Base model
print("\n1. Base Model (no LoRA):")
profile_memory(model, batch_size=4)

# With LoRA
print("\n2. With LoRA (r=16):")
lora_for_profile = get_peft_model(
    AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32).cuda(),
    create_lora_config(rank=16)
)
profile_memory(lora_for_profile, batch_size=4)

## 6. Complete PEFT Pipeline

In [ ]:
class PEFTPipeline:
    """Complete PEFT training pipeline."""
    
    def __init__(self, model_name: str, use_qlora: bool = False):
        self.model_name = model_name
        self.use_qlora = use_qlora
        self.model = None
        self.tokenizer = None
        self.history = None
    
    def load_model(self, rank=16, alpha=None, dropout=0.05):
        """Load model with LoRA/QLoRA."""
        print(f"Loading model: {self.model_name}")
        
        if self.use_qlora and BNB_AVAILABLE:
            # QLoRA setup
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
            
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_config,
                device_map="auto",
            )
            self.model = prepare_model_for_kbit_training(self.model)
            print("Loaded with 4-bit quantization (QLoRA)")
        else:
            # Standard LoRA
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
            )
            print("Loaded in 16-bit (LoRA)")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Apply LoRA
        lora_config = create_lora_config(rank=rank, alpha=alpha, dropout=dropout)
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()
        
        return self
    
    def load_data(self, dataset_name: str, max_samples: int = 500):
        """Load and tokenize dataset."""
        print(f"\nLoading dataset: {dataset_name}")
        dataset = load_dataset(dataset_name, split=f"train[:{max_samples}]")
        
        def tokenize(example):
            return self.tokenizer(
                example.get("text", str(example)),
                truncation=True,
                max_length=256,
                padding="max_length"
            )
        
        self.tokenized_dataset = dataset.map(tokenize, batched=True)
        self.tokenized_dataset.set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"]
        )
        
        print(f"Loaded {len(self.tokenized_dataset)} samples")
        return self
    
    def train(self, output_dir: str = "./peft-output", **kwargs):
        """Train the model."""
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=kwargs.get("batch_size", 4),
            gradient_accumulation_steps=kwargs.get("grad_accum", 4),
            learning_rate=kwargs.get("lr", 2e-4),
            max_steps=kwargs.get("max_steps", 100),
            logging_steps=kwargs.get("log_steps", 10),
            report_to="none",
            fp16=False,
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.tokenized_dataset,
        )
        
        print("\nStarting training...")
        trainer.train()
        self.history = trainer.state.log_history
        
        trainer.save_model(output_dir)
        print(f"Model saved to {output_dir}")
        
        return self
    
    def plot_training(self):
        """Plot training loss curve."""
        if not self.history:
            print("No training history available")
            return
        
        loss_logs = [log for log in self.history if 'loss' in log]
        steps = [log['step'] for log in loss_logs]
        losses = [log['loss'] for log in loss_logs]
        
        plt.figure(figsize=(10, 5))
        plt.plot(steps, losses, 'go-', linewidth=2, markersize=6)
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('Training Loss Curve')
        plt.grid(True, alpha=0.3)
        
        # Add trend line
        if len(losses) > 2:
            z = np.polyfit(steps, losses, 2)
            p = np.poly1d(z)
            plt.plot(steps, p(steps), 'r--', alpha=0.5, label='Trend')
            plt.legend()
        
        plt.tight_layout()
        plt.show()
    
    def generate(self, prompt: str, max_length: int = 100):
        """Generate text with the model."""
        self.model.eval()
        inputs = self.tokenizer(prompt, return_tensors="pt")
        
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_length,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )
        
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"\nPrompt: {prompt}")
        print(f"Generated: {generated}")
        return generated


# Example usage
print("=" * 60)
print("PEFT Pipeline Demo")
print("=" * 60)

pipeline = PEFTPipeline(model_name, use_qlora=False)
pipeline.load_model(rank=8)
pipeline.load_data("mlabonne/FineTome-100k", max_samples=200)
pipeline.train(output_dir="./peft-demo", batch_size=2, max_steps=30)
pipeline.plot_training()

# Test generation
print("\n" + "=" * 60)
pipeline.generate("What is machine learning?")

## Summary

This notebook covered:

1. **LoRA Configuration**: Rank, alpha, dropout parameters
2. **QLoRA**: 4-bit quantization for memory efficiency
3. **Rank Ablation**: Finding optimal rank for your dataset
4. **Multi-Adapter**: Multiple adapters, merging for deployment
5. **Memory Profiling**: Understanding VRAM usage
6. **Complete Pipeline**: End-to-end PEFT training class

### Key Takeaways:

- LoRA reduces trainable params by 100× (0.1-0.5% of total)
- QLoRA enables 7B fine-tuning on ~3GB VRAM
- Start with r=16, α=32, adjust based on dataset size
- Merge adapters before deployment for inference efficiency
- Memory profiling helps optimize batch size